In [ ]:
from pathlib import Path
import os

def _analysis_data_root():
    if os.environ.get("ANALYSIS_DATA_DIR"):
        return Path(os.environ["ANALYSIS_DATA_DIR"]).expanduser().resolve()
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base / "analysis_data", base / "A_paper" / "analysis_data"):
            if candidate.is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set ANALYSIS_DATA_DIR to the extracted analysis_data directory")

DATA_ROOT = _analysis_data_root()
RESULTS_ROOT = Path(os.environ.get("ANALYSIS_RESULTS_DIR", DATA_ROOT.parent / "analysis_code" / "results")).expanduser().resolve()


## Figure caption: Kruskal–Wallis test followed by BH-adjusted pairwise Mann–Whitney U tests versus Q4.

Full: Group differences were assessed using a Kruskal–Wallis test, followed by pairwise Mann–Whitney U tests comparing Q1, Q2, and Q3 with Q4, with Benjamini–Hochberg correction for multiple testing.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Batch cross-species RNA–ATAC analysis: human RNA → macaque ATAC

Deliverable scope
-----------------
Only the all-peaks contrast is analysed, because that is what the paper's
Figure 6g-6h reports. The alternative top-5000 attention peak set is
restricted to the all-peak metagene profile; ranked peak scores are merged
for annotation but no rank-depth or top-peak subset is analysed.

Core analysis
-------------
Human cell-type-specific gene attention
    -> mapped macaque ortholog genes
    -> macaque Mmul_10 gene coordinates
    -> macaque matched-cell-type ATAC accessibility
    -> macaque matched-cell-type peak-attention ranking

Auxiliary analysis
------------------
Human cell-type-specific gene attention
    -> human matched-cell-type RNA expression

Important input interpretation
------------------------------
1. Cell-type-specific macaque ortholog-gene CSV
   gene_id      : macaque Ensembl gene ID, e.g. ENSMMUG00000021442
   gene_symbol  : macaque gene symbol
   score        : gene-attention score inherited from the corresponding human cell-type analysis

2. Cell-type-specific human gene-rank CSV
   gene_id      : human Ensembl gene ID
   gene_symbol  : human gene symbol
   score        : human cell-type-specific gene-attention score

3. Cell-type-specific macaque peak-rank CSV
   expected columns:
       peak, score
   where peak is formatted as chr1:100-200 or chr1_100_200.

Main figures
------------
A. Human gene-attention quartiles vs macaque ATAC accessibility, all peaks:
   - metagene profile: upstream 100 kb -> TSS -> scaled gene body -> TES -> +100 kb

B. Human gene-attention quartiles vs matched-cell-type human RNA expression:
   - jitter + boxplot
   - Kruskal-Wallis test
   - BH-adjusted Mann-Whitney U tests against Q4

Notes
-----
- Every RNA × ATAC sample combination is processed for the cell types listed in
  CELL_TYPES; the paper reports the ASC contrast.
- PDFs use fonttype 42, so text remains editable.
"""

from __future__ import annotations

import gc
import gzip
import math
import re
import sys
from pathlib import Path
from typing import Dict, Iterable, Optional
from urllib.parse import unquote

import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats


# ============================================================
# 1. Paths and paired analyses
# ============================================================

BASE_DIR = DATA_ROOT / "Figure6/brain"

RANK_DIR = BASE_DIR / "rankings"
MAPPED_RNA_RANK_DIR = RESULTS_ROOT / "Figure6/brain/human_rna_to_macaca/converted"
if not MAPPED_RNA_RANK_DIR.is_dir() or not any(MAPPED_RNA_RANK_DIR.glob("*.csv")):
    MAPPED_RNA_RANK_DIR = RANK_DIR / "human_rna_to_macaca" / "converted"

MACAQUE_GFF_FILE = BASE_DIR / "Mmul10_genomic.gff.gz"

# Build all RNA × ATAC combinations within each cell type.
#
# One human RNA sample × one macaque ATAC sample × ASC
# = one main-figure analysis.
#
# ASC is only paired with ASC rank files, and ODC is only paired with ODC
# rank files. Cell types are never mixed.
RNA_SAMPLES = {
    "M1donor": BASE_DIR / "homo_sapiens_M1donor_rna.h5ad",
}

ATAC_SAMPLES = {
    "M1mac3": BASE_DIR / "Macaca_mulatta_M1mac3_atac.h5ad",
}

# The paper's Figure 6g-6h reports the astrocyte-lineage (ASC) contrast, so the
# standard run analyses that cell type only. The other three labels were analysed
# during development and their output directories are not part of the paper's
# deliverable set; add them back to this list to regenerate them.
CELL_TYPES = ["ASC"]

def sanitize_label(value: str) -> str:
    """Convert a biological label into a filesystem-safe label."""
    value = str(value).strip()
    value = re.sub(r"[\\/\s]+", "_", value)
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


ANALYSIS_CONFIGS = []

for cell_type in CELL_TYPES:
    cell_type_file_label = sanitize_label(cell_type)

    for rna_sample, human_rna_h5ad_file in RNA_SAMPLES.items():
        for atac_sample, macaque_atac_h5ad_file in ATAC_SAMPLES.items():
            ANALYSIS_CONFIGS.append(
                {
                    "analysis_name": (
                        f"RNA_{rna_sample}__ATAC_{atac_sample}__"
                        f"{cell_type_file_label}"
                    ),
                    "cell_type": cell_type,
                    "cell_type_file_label": cell_type_file_label,
                    "rna_sample": rna_sample,
                    "atac_sample": atac_sample,
                    "human_gene_rank_file": (
                        RANK_DIR
                        / (
                            "motc_homo_sapiens_"
                            f"{rna_sample}_rna_{cell_type_file_label}.csv"
                        )
                    ),
                    "macaque_gene_rank_file": (
                        MAPPED_RNA_RANK_DIR
                        / (
                            "motc_Macaca_mulatta_"
                            f"{rna_sample}_rna_{cell_type_file_label}.csv"
                        )
                    ),
                    "macaque_peak_rank_file": (
                        RANK_DIR
                        / (
                            "motc_Macaca_mulatta_"
                            f"{atac_sample}_atac_{cell_type_file_label}.csv"
                        )
                    ),
                    "human_rna_h5ad_file": human_rna_h5ad_file,
                    "macaque_atac_h5ad_file": macaque_atac_h5ad_file,
                }
            )

BATCH_OUTPUT_DIR = RESULTS_ROOT / "Figure6/brain/gene_peak_accessibility"
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Cell-type settings
# ============================================================

CELLTYPE_COLUMN = "Celltype"

CELL_TYPE_ALIASES = {
    "OPC": {
        "OPC",
    },
    "ASC": {
        "ASC",
    },
    "ODC": {
        "ODC",
    },
    "L2/3 IT": {
        "L2/3 IT",
        "L2_3_IT",
    },
}

CELL_TYPE_DISPLAY_NAMES = {
    "OPC": "OPC",
    "ASC": "Astrocyte",
    "ODC": "Oligodendrocyte",
    "L2/3 IT": "L2/3 IT neuron",
}

# These globals are configured separately for every pair.
CURRENT_ANALYSIS_NAME = ""
CURRENT_CELL_TYPE = ""
CURRENT_RNA_SAMPLE = ""
CURRENT_ATAC_SAMPLE = ""
CURRENT_CELL_TYPE_DISPLAY = ""
CURRENT_CELL_TYPE_FILE_LABEL = ""
CURRENT_CELLTYPE_ALIASES: set[str] = set()

MACAQUE_GENE_RANK_FILE: Path
HUMAN_GENE_RANK_FILE: Path
MACAQUE_PEAK_RANK_FILE: Path
MACAQUE_ATAC_H5AD_FILE: Path
HUMAN_RNA_H5AD_FILE: Path
OUTPUT_DIR: Path


# ============================================================
# 3. Count layers and normalization
# ============================================================

ATAC_COUNTS_LAYER = "counts"
RNA_COUNTS_LAYER = "counts"

ATAC_TARGET_SUM = 10_000
RNA_TARGET_SUM = 10_000

# ATAC metric used as the main experimental accessibility signal.
# Choices:
#   mean_normalized
#   fraction_accessible
#   pseudobulk_cpm
#   mean_raw_count
ACCESSIBILITY_METRIC = "mean_normalized"
LOG1P_ACCESSIBILITY = False

RNA_EXPRESSION_COLUMN = "human_ex_mean_log_normalized_expression"


# ============================================================
# 4. Peak and gene grouping
# ============================================================

N_GENE_QUANTILES = 4

# Main metagene:
#   upstream 100 kb -> TSS -> scaled gene body -> TES -> downstream 100 kb
FLANK_SIZE = 100_000
UPSTREAM_BINS = 100
GENE_BODY_BINS = 100
DOWNSTREAM_BINS = 100

SMOOTH_PROFILE = True
SMOOTH_WINDOW = 7
SHOW_SEM = True

# Macaca mulatta Mmul_10 standard chromosomes
MACAQUE_STANDARD_CHROMS = {
    *(f"chr{i}" for i in range(1, 21)),
    "chrX",
    "chrY",
    "chrM",
}

KEEP_STANDARD_CHROMOSOMES = True
EXCLUDE_MITOCHONDRIAL = True


# ============================================================
# 5. Plot settings
# ============================================================

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["axes.unicode_minus"] = False
# mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8


# ============================================================
# 6. Output paths
# ============================================================

MAPPED_GENE_COORDINATE_OUTPUT: Path
UNMATCHED_GENE_OUTPUT: Path
DUPLICATE_GENE_OUTPUT: Path
SEQID_CHROMOSOME_MAP_OUTPUT: Path
ALL_PEAK_ACCESSIBILITY_OUTPUT: Path
UNMATCHED_PEAK_RANK_OUTPUT: Path
ALL_PROFILE_MATRIX_OUTPUT: Path
ALL_PROFILE_SUMMARY_OUTPUT: Path
ALL_PROFILE_FIGURE_PREFIX: Path
HUMAN_RNA_GENE_OUTPUT: Path
HUMAN_RNA_SUMMARY_OUTPUT: Path
HUMAN_RNA_STATS_OUTPUT: Path
HUMAN_RNA_FIGURE_PREFIX: Path


def configure_output_paths() -> None:
    global MAPPED_GENE_COORDINATE_OUTPUT, UNMATCHED_GENE_OUTPUT, DUPLICATE_GENE_OUTPUT, SEQID_CHROMOSOME_MAP_OUTPUT, ALL_PEAK_ACCESSIBILITY_OUTPUT, UNMATCHED_PEAK_RANK_OUTPUT, ALL_PROFILE_MATRIX_OUTPUT, ALL_PROFILE_SUMMARY_OUTPUT, ALL_PROFILE_FIGURE_PREFIX, HUMAN_RNA_GENE_OUTPUT, HUMAN_RNA_SUMMARY_OUTPUT, HUMAN_RNA_STATS_OUTPUT, HUMAN_RNA_FIGURE_PREFIX
    MAPPED_GENE_COORDINATE_OUTPUT = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_macaque_genes_with_Mmul10_coordinates.csv"
    )
    UNMATCHED_GENE_OUTPUT = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_macaque_genes_unmatched_to_Mmul10_GFF.csv"
    )
    DUPLICATE_GENE_OUTPUT = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_macaque_gene_duplicates_removed.csv"
    )
    SEQID_CHROMOSOME_MAP_OUTPUT = (
        OUTPUT_DIR / "Mmul10_seqid_to_chromosome.csv"
    )
    
    ALL_PEAK_ACCESSIBILITY_OUTPUT = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_macaque_all_peak_accessibility.csv"
    )
    UNMATCHED_PEAK_RANK_OUTPUT = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_macaque_ranked_peaks_unmatched_to_ATAC.csv"
    )
    
    ALL_PROFILE_MATRIX_OUTPUT = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_gene_attention_all_peaks_gene_bin_matrix.csv"
    )
    ALL_PROFILE_SUMMARY_OUTPUT = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_gene_attention_all_peaks_summary.csv"
    )
    
    ALL_PROFILE_FIGURE_PREFIX = (
        OUTPUT_DIR / f"{CURRENT_CELL_TYPE_FILE_LABEL}_gene_attention_all_peaks_profile"
    )
    
    HUMAN_RNA_GENE_OUTPUT = (
        OUTPUT_DIR / f"human_{CURRENT_CELL_TYPE_FILE_LABEL}_gene_attention_RNA_expression_genes.csv"
    )
    HUMAN_RNA_SUMMARY_OUTPUT = (
        OUTPUT_DIR / f"human_{CURRENT_CELL_TYPE_FILE_LABEL}_gene_attention_RNA_expression_summary.csv"
    )
    HUMAN_RNA_STATS_OUTPUT = (
        OUTPUT_DIR / f"human_{CURRENT_CELL_TYPE_FILE_LABEL}_gene_attention_RNA_expression_pairwise_stats.csv"
    )
    HUMAN_RNA_FIGURE_PREFIX = (
        OUTPUT_DIR / f"human_{CURRENT_CELL_TYPE_FILE_LABEL}_gene_attention_RNA_expression"
    )
    
    
    

# ============================================================
# 7. General helpers
# ============================================================

def require_file(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")


def strip_ensembl_version(value: object) -> str:
    return re.sub(r"\.\d+$", "", str(value).strip())


def clean_string(value: object) -> str:
    value = str(value).strip()
    if value.lower() in {"", "nan", "none", "na"}:
        return ""
    return value


def rolling_smooth(values: np.ndarray, window: int) -> np.ndarray:
    if window <= 1:
        return values.copy()

    return (
        pd.Series(values)
        .rolling(window=window, center=True, min_periods=1)
        .mean()
        .to_numpy()
    )


def assign_quantiles(
    frame: pd.DataFrame,
    value_column: str,
    output_column: str,
    n_quantiles: int = N_GENE_QUANTILES,
) -> pd.DataFrame:
    result = frame.dropna(subset=[value_column]).copy()

    if len(result) < n_quantiles:
        raise ValueError(
            f"{value_column} has only {len(result)} valid rows; "
            f"cannot divide into {n_quantiles} quantiles."
        )

    ranked = result[value_column].rank(method="first", ascending=True)

    result[output_column] = pd.qcut(
        ranked,
        q=n_quantiles,
        labels=[f"Q{i}" for i in range(1, n_quantiles + 1)],
    ).astype(str)

    return result


def pvalue_to_stars(p_value: float) -> str:
    if pd.isna(p_value):
        return "ns"
    if p_value < 1e-4:
        return "****"
    if p_value < 1e-3:
        return "***"
    if p_value < 1e-2:
        return "**"
    if p_value < 5e-2:
        return "*"
    return "ns"


def benjamini_hochberg(p_values: Iterable[float]) -> np.ndarray:
    values = np.asarray(list(p_values), dtype=float)
    n = len(values)

    if n == 0:
        return np.array([], dtype=float)

    order = np.argsort(values)
    ranked = values[order]
    adjusted_sorted = np.empty(n, dtype=float)

    running_min = 1.0
    for index in range(n - 1, -1, -1):
        rank = index + 1
        candidate = ranked[index] * n / rank
        running_min = min(running_min, candidate)
        adjusted_sorted[index] = running_min

    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.clip(adjusted_sorted, 0, 1)
    return adjusted


# ============================================================
# 8. Target-cell selection
# ============================================================

def select_excitatory_mask(
    obs: pd.DataFrame,
    celltype_column: str,
    dataset_name: str,
) -> np.ndarray:
    if celltype_column not in obs.columns:
        raise KeyError(
            f"{dataset_name}.obs does not contain column: {celltype_column}"
        )

    labels = obs[celltype_column].astype(str).str.strip()
    normalized = labels.str.lower()

    print(f"\n{dataset_name} cell-type labels:")
    print(labels.value_counts(dropna=False).to_string())

    subtype_table = (
        pd.DataFrame(
            {
                "celltype": labels,
                "normalized": normalized,
            }
        )
        .groupby(["celltype", "normalized"], observed=True)
        .size()
        .reset_index(name="n_cells")
    )

    subtype_table["selected_as_target"] = subtype_table[
        "normalized"
    ].isin(CURRENT_CELLTYPE_ALIASES)

    print(f"\n{dataset_name} requested target cell-type counts:")
    requested_rows = subtype_table.loc[subtype_table["selected_as_target"]]
    if requested_rows.empty:
        print("  No requested target labels matched.")
    else:
        print(
            requested_rows[
                ["celltype", "n_cells"]
            ].sort_values("celltype").to_string(index=False)
        )

    mask = normalized.isin(CURRENT_CELLTYPE_ALIASES).to_numpy()
    n_selected = int(mask.sum())

    if n_selected == 0:
        raise ValueError(
            f"No target cells were selected in {dataset_name}. "
            f"Check CELL_TYPE_ALIASES and {celltype_column}."
        )

    print(
        f"{dataset_name} selected target cells: "
        f"{n_selected:,}/{len(obs):,}"
    )

    return mask


# ============================================================
# 9. Gene-rank loading
# ============================================================

def load_ranked_genes(
    path: Path,
    species_prefix: str,
    score_output_name: str,
) -> pd.DataFrame:
    require_file(path)

    frame = pd.read_csv(path)

    frame = frame.copy()
    frame.columns = frame.columns.astype(str).str.strip()

    if "gene_id" not in frame.columns and "gene" in frame.columns:
        frame = frame.rename(columns={"gene": "gene_id"})

    if "gene_symbol" not in frame.columns:
        frame["gene_symbol"] = ""

    required = {"gene_id", "score"}
    missing = required - set(frame.columns)

    if missing:
        raise ValueError(
            f"{path} is missing columns: {sorted(missing)}. "
            f"Current columns: {frame.columns.tolist()}"
        )

    frame["gene_id"] = (
        frame["gene_id"]
        .astype(str)
        .map(strip_ensembl_version)
    )
    frame["gene_symbol"] = (
        frame["gene_symbol"]
        .astype(str)
        .str.strip()
    )
    frame["gene_symbol_upper"] = (
        frame["gene_symbol"]
        .str.upper()
    )
    frame[score_output_name] = pd.to_numeric(
        frame["score"],
        errors="coerce",
    )

    frame = frame.dropna(
        subset=[
            "gene_id",
            "gene_symbol",
            score_output_name,
        ]
    ).copy()

    frame = frame[
        frame["gene_id"].astype(str).str.len() > 0
    ].copy()

    duplicate_mask = frame.duplicated(
        subset=["gene_id"],
        keep=False,
    )

    if duplicate_mask.any():
        duplicate_rows = (
            frame.loc[duplicate_mask]
            .sort_values(
                ["gene_id", score_output_name],
                ascending=[True, False],
            )
        )

        duplicate_file = (
            DUPLICATE_GENE_OUTPUT
            if species_prefix == "macaque"
            else OUTPUT_DIR / "human_gene_duplicates_removed.csv"
        )

        duplicate_rows.to_csv(
            duplicate_file,
            index=False,
        )

        print(
            f"Warning: duplicated {species_prefix} gene IDs: "
            f"{duplicate_mask.sum():,} rows. "
            f"Saved: {duplicate_file}"
        )

    frame = (
        frame.sort_values(
            score_output_name,
            ascending=False,
        )
        .drop_duplicates(
            subset=["gene_id"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    frame[f"{species_prefix}_gene_rank"] = (
        np.arange(1, len(frame) + 1)
    )

    return frame[
        [
            "gene_id",
            "gene_symbol",
            "gene_symbol_upper",
            score_output_name,
            f"{species_prefix}_gene_rank",
        ]
    ]


# ============================================================
# 10. GFF3 parsing
# ============================================================

def parse_gff3_attributes(attribute_text: str) -> Dict[str, str]:
    attributes: Dict[str, str] = {}

    for item in str(attribute_text).strip().strip(";").split(";"):
        item = item.strip()

        if not item or "=" not in item:
            continue

        key, value = item.split("=", 1)
        attributes[key.strip()] = unquote(value.strip())

    return attributes


def chromosome_label_from_attribute(value: object) -> Optional[str]:
    text = clean_string(value)

    if not text:
        return None

    upper = text.upper().replace("CHR", "")

    if upper in {"M", "MT", "MITOCHONDRION", "MITOCHONDRIAL"}:
        return "chrM"

    if upper in {"X", "Y"}:
        return f"chr{upper}"

    if upper.isdigit():
        number = int(upper)
        if 1 <= number <= 20:
            return f"chr{number}"

    return None


def normalize_macaque_chromosome(
    value: object,
    seqid_to_chrom: Optional[Dict[str, str]] = None,
) -> Optional[str]:
    text = clean_string(value)

    if not text:
        return None

    if seqid_to_chrom is not None and text in seqid_to_chrom:
        return seqid_to_chrom[text]

    if text.startswith("chr"):
        suffix = text[3:]

        if suffix.upper() in {"M", "MT"}:
            return "chrM"

        if suffix.upper() in {"X", "Y"}:
            return f"chr{suffix.upper()}"

        if suffix.isdigit() and 1 <= int(suffix) <= 20:
            return f"chr{int(suffix)}"

        return text

    upper = text.upper()

    if upper in {"M", "MT"}:
        return "chrM"

    if upper in {"X", "Y"}:
        return f"chr{upper}"

    if upper.isdigit() and 1 <= int(upper) <= 20:
        return f"chr{int(upper)}"

    return text


def extract_ensembl_gene_ids(attributes: Dict[str, str]) -> list[str]:
    candidates: list[str] = []

    for key in [
        "gene_id",
        "GeneID",
        "ensembl_gene_id",
        "gene",
        "ID",
        "Dbxref",
    ]:
        value = clean_string(attributes.get(key, ""))

        if not value:
            continue

        matches = re.findall(
            r"ENSMMUG\d+(?:\.\d+)?",
            value,
            flags=re.IGNORECASE,
        )

        candidates.extend(
            strip_ensembl_version(match.upper())
            for match in matches
        )

    return sorted(set(candidates))


def extract_gene_symbol(attributes: Dict[str, str]) -> str:
    for key in [
        "gene",
        "gene_name",
        "Name",
        "symbol",
        "locus_tag",
    ]:
        value = clean_string(attributes.get(key, ""))

        if value:
            # Avoid treating Ensembl ID as a symbol.
            if re.fullmatch(
                r"ENSMMUG\d+(?:\.\d+)?",
                value,
                flags=re.IGNORECASE,
            ):
                continue

            return value

    return ""


def load_macaque_gene_coordinates_from_gff(
    path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    require_file(path)

    opener = gzip.open if path.suffix == ".gz" else open

    # First pass: infer sequence-ID -> chromosome label from region features.
    seqid_to_chrom: Dict[str, str] = {}
    region_records: list[dict] = []

    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            if not line or line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            seqid, _, feature_type, start, end, _, strand, _, attr_text = fields

            if feature_type != "region":
                continue

            attrs = parse_gff3_attributes(attr_text)

            chromosome = (
                chromosome_label_from_attribute(attrs.get("chromosome"))
                or chromosome_label_from_attribute(attrs.get("Name"))
                or chromosome_label_from_attribute(attrs.get("ID"))
            )

            if chromosome is None:
                continue

            seqid_to_chrom[seqid] = chromosome

            region_records.append(
                {
                    "seqid": seqid,
                    "chrom": chromosome,
                    "region_start_1based": start,
                    "region_end_1based": end,
                    "region_name": attrs.get("Name", ""),
                    "region_id": attrs.get("ID", ""),
                }
            )

    chromosome_map = pd.DataFrame(region_records)

    if chromosome_map.empty:
        print(
            "Warning: no chromosome map was inferred from GFF region features. "
            "Peak/GFF chromosome overlap will be checked explicitly."
        )
    else:
        chromosome_map = (
            chromosome_map
            .drop_duplicates(subset=["seqid"], keep="first")
            .sort_values("chrom")
            .reset_index(drop=True)
        )

    # Second pass: parse gene features.
    records: list[dict] = []

    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            if not line or line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            if len(fields) != 9:
                continue

            seqid, source, feature_type, start, end, score, strand, phase, attr_text = fields

            if feature_type != "gene":
                continue

            if strand not in {"+", "-"}:
                continue

            try:
                start_1based = int(start)
                end_1based = int(end)
            except ValueError:
                continue

            if end_1based < start_1based:
                continue

            attrs = parse_gff3_attributes(attr_text)

            chrom = normalize_macaque_chromosome(
                seqid,
                seqid_to_chrom=seqid_to_chrom,
            )

            gene_start = start_1based - 1
            gene_end = end_1based

            if gene_end <= gene_start:
                continue

            tss = gene_start if strand == "+" else gene_end
            tes = gene_end if strand == "+" else gene_start

            ensembl_ids = extract_ensembl_gene_ids(attrs)
            gene_symbol = extract_gene_symbol(attrs)

            records.append(
                {
                    "gff_seqid": seqid,
                    "chrom": chrom,
                    "gene_start": gene_start,
                    "gene_end": gene_end,
                    "strand": strand,
                    "tss": tss,
                    "tes": tes,
                    "gene_length": gene_end - gene_start,
                    "gff_gene_symbol": gene_symbol,
                    "gene_symbol_upper": gene_symbol.upper(),
                    "gff_ensembl_gene_ids": ",".join(ensembl_ids),
                    "gff_primary_ensembl_gene_id": (
                        ensembl_ids[0] if ensembl_ids else ""
                    ),
                    "gff_feature_id": attrs.get("ID", ""),
                    "gff_name": attrs.get("Name", ""),
                    "gff_gene_biotype": (
                        attrs.get("gene_biotype")
                        or attrs.get("gene_type")
                        or attrs.get("biotype")
                        or ""
                    ),
                    "gff_source": source,
                }
            )

    coordinates = pd.DataFrame(records)

    if coordinates.empty:
        raise ValueError(f"No gene features were parsed from GFF: {path}")

    coordinates["is_standard_chromosome"] = coordinates[
        "chrom"
    ].isin(MACAQUE_STANDARD_CHROMS)

    coordinates = (
        coordinates.sort_values(
            [
                "gene_symbol_upper",
                "is_standard_chromosome",
                "gene_length",
            ],
            ascending=[True, False, False],
        )
        .reset_index(drop=True)
    )

    return coordinates, chromosome_map


# ============================================================
# 11. Match mapped macaque genes to Mmul_10 GFF
# ============================================================

def match_macaque_genes_to_gff(
    ranked_genes: pd.DataFrame,
    coordinates: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    id_lookup: Dict[str, list[int]] = {}
    symbol_lookup: Dict[str, list[int]] = {}

    for index, row in coordinates.iterrows():
        ids = [
            item
            for item in str(row["gff_ensembl_gene_ids"]).split(",")
            if item
        ]

        primary_id = clean_string(row["gff_primary_ensembl_gene_id"])
        if primary_id:
            ids.append(primary_id)

        for gene_id in set(ids):
            id_lookup.setdefault(
                strip_ensembl_version(gene_id),
                [],
            ).append(index)

        symbol = clean_string(row["gene_symbol_upper"])
        if symbol:
            symbol_lookup.setdefault(symbol, []).append(index)

    matched_records: list[dict] = []
    unmatched_records: list[dict] = []

    for gene in ranked_genes.itertuples(index=False):
        macaque_gene_id = strip_ensembl_version(gene.gene_id)
        symbol_upper = clean_string(gene.gene_symbol_upper)

        matched_indices: list[int] = []
        match_method = ""

        if macaque_gene_id in id_lookup:
            matched_indices = id_lookup[macaque_gene_id]
            match_method = "ensembl_gene_id"
        elif symbol_upper in symbol_lookup:
            matched_indices = symbol_lookup[symbol_upper]
            match_method = "gene_symbol"

        if not matched_indices:
            unmatched_records.append(
                {
                    "macaque_gene_id": macaque_gene_id,
                    "macaque_gene_symbol": gene.gene_symbol,
                    "human_gene_attention": gene.human_gene_attention,
                    "macaque_gene_rank": gene.macaque_gene_rank,
                }
            )
            continue

        candidates = coordinates.loc[matched_indices].copy()

        candidates = candidates.sort_values(
            ["is_standard_chromosome", "gene_length"],
            ascending=[False, False],
        )

        selected = candidates.iloc[0]

        matched_records.append(
            {
                "macaque_gene_id": macaque_gene_id,
                "macaque_gene_symbol": gene.gene_symbol,
                "human_gene_attention": gene.human_gene_attention,
                "macaque_gene_rank": gene.macaque_gene_rank,
                "gff_match_method": match_method,
                "gff_n_candidate_records": len(candidates),
                **selected.to_dict(),
            }
        )

    matched = pd.DataFrame(matched_records)
    unmatched = pd.DataFrame(unmatched_records)

    if matched.empty:
        raise ValueError(
            "No mapped macaque genes matched the Mmul_10 GFF. "
            "Inspect GFF attributes and gene symbols."
        )

    matched = assign_quantiles(
        frame=matched,
        value_column="human_gene_attention",
        output_column="gene_attention_quantile",
    )

    matched = (
        matched.sort_values(
            "human_gene_attention",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return matched, unmatched


# ============================================================
# 12. Peak parsing and rank loading
# ============================================================

def parse_peak_intervals(intervals: pd.Series) -> pd.DataFrame:
    strings = intervals.astype(str).str.strip()

    parsed = strings.str.extract(
        r"^(.+?):(\d+)-(\d+)$"
    )
    parsed.columns = ["chrom", "peak_start", "peak_end"]

    unresolved = parsed.isna().any(axis=1)

    if unresolved.any():
        fallback = strings.loc[unresolved].str.extract(
            r"^(.+?)_(\d+)_(\d+)$"
        )
        fallback.columns = ["chrom", "peak_start", "peak_end"]

        parsed.loc[
            unresolved,
            ["chrom", "peak_start", "peak_end"],
        ] = fallback.values

    parsed["chrom"] = parsed["chrom"].map(
        normalize_macaque_chromosome
    )
    parsed["peak_start"] = pd.to_numeric(
        parsed["peak_start"],
        errors="coerce",
    )
    parsed["peak_end"] = pd.to_numeric(
        parsed["peak_end"],
        errors="coerce",
    )

    parsed["peak_id_canonical"] = np.where(
        parsed[["chrom", "peak_start", "peak_end"]]
        .notna()
        .all(axis=1),
        (
            parsed["chrom"].astype(str)
            + ":"
            + parsed["peak_start"].fillna(-1).astype(int).astype(str)
            + "-"
            + parsed["peak_end"].fillna(-1).astype(int).astype(str)
        ),
        "",
    )

    return parsed


def load_ranked_peaks(path: Path) -> pd.DataFrame:
    require_file(path)

    frame = pd.read_csv(path)

    peak_column_candidates = [
        "peak",
        "peak_id",
        "interval",
        "region",
        "gene",  # peak coordinates in the unsupervised rank export
    ]
    score_column_candidates = [
        "score",
        "peak_attention",
        "attention_score",
        "peak_score",
    ]

    peak_column = next(
        (
            column
            for column in peak_column_candidates
            if column in frame.columns
        ),
        None,
    )
    score_column = next(
        (
            column
            for column in score_column_candidates
            if column in frame.columns
        ),
        None,
    )

    if peak_column is None or score_column is None:
        raise ValueError(
            f"{path} must contain a peak column and a score column. "
            f"Current columns: {frame.columns.tolist()}"
        )

    frame = frame.copy()

    frame["peak_id_input"] = frame[peak_column].astype(str).str.strip()
    frame["peak_attention"] = pd.to_numeric(
        frame[score_column],
        errors="coerce",
    )

    parsed = parse_peak_intervals(frame["peak_id_input"])

    frame = pd.concat(
        [
            frame.reset_index(drop=True),
            parsed.reset_index(drop=True),
        ],
        axis=1,
    )

    frame = frame.dropna(
        subset=[
            "chrom",
            "peak_start",
            "peak_end",
            "peak_attention",
        ]
    ).copy()

    frame["peak_start"] = frame["peak_start"].astype(int)
    frame["peak_end"] = frame["peak_end"].astype(int)

    frame = frame[
        frame["peak_end"] > frame["peak_start"]
    ].copy()

    frame = (
        frame.sort_values(
            "peak_attention",
            ascending=False,
        )
        .drop_duplicates(
            subset=["peak_id_canonical"],
            keep="first",
        )
        .reset_index(drop=True)
    )

    frame["peak_attention_rank"] = (
        np.arange(1, len(frame) + 1)
    )

    return frame[
        [
            "peak_id_input",
            "peak_id_canonical",
            "chrom",
            "peak_start",
            "peak_end",
            "peak_attention",
            "peak_attention_rank",
        ]
    ]


# ============================================================
# 13. Macaque target-cell ATAC accessibility
# ============================================================

def calculate_macaque_ex_peak_accessibility(
    adata_atac: ad.AnnData,
) -> pd.DataFrame:
    if ATAC_COUNTS_LAYER not in adata_atac.layers:
        raise KeyError(
            f"ATAC layers does not contain: {ATAC_COUNTS_LAYER}"
        )

    ex_mask = select_excitatory_mask(
        obs=adata_atac.obs,
        celltype_column=CELLTYPE_COLUMN,
        dataset_name="Macaque ATAC",
    )

    counts = adata_atac.layers[ATAC_COUNTS_LAYER][ex_mask, :]

    if sp.issparse(counts):
        counts = counts.tocsr().astype(np.float64)

        nonzero_cell_count = np.asarray(
            (counts > 0).sum(axis=0)
        ).ravel()
        pseudobulk_count = np.asarray(
            counts.sum(axis=0)
        ).ravel()
        mean_raw_count = np.asarray(
            counts.mean(axis=0)
        ).ravel()
        cell_totals = np.asarray(
            counts.sum(axis=1)
        ).ravel()
    else:
        counts = np.asarray(counts, dtype=np.float64)

        nonzero_cell_count = (counts > 0).sum(axis=0)
        pseudobulk_count = counts.sum(axis=0)
        mean_raw_count = counts.mean(axis=0)
        cell_totals = counts.sum(axis=1)

    n_cells = int(ex_mask.sum())

    fraction_accessible = nonzero_cell_count / n_cells

    total_fragments = float(pseudobulk_count.sum())

    if total_fragments <= 0:
        raise ValueError(f"Selected macaque {CURRENT_CELL_TYPE_DISPLAY} ATAC counts sum to zero.")

    pseudobulk_cpm = (
        pseudobulk_count
        / total_fragments
        * 1e6
    )

    scale_factors = np.divide(
        ATAC_TARGET_SUM,
        cell_totals,
        out=np.zeros_like(cell_totals, dtype=float),
        where=cell_totals > 0,
    )

    if sp.issparse(counts):
        mean_normalized = np.asarray(
            counts.T.dot(scale_factors)
        ).ravel() / n_cells
    else:
        mean_normalized = (
            counts.T @ scale_factors
        ) / n_cells

    if "interval" in adata_atac.var.columns:
        peak_ids = (
            adata_atac.var["interval"]
            .astype(str)
            .to_numpy()
        )
    elif "gene_ids" in adata_atac.var.columns:
        # In ATAC AnnData, gene_ids is often used to store peak intervals.
        candidate = (
            adata_atac.var["gene_ids"]
            .astype(str)
            .to_numpy()
        )

        parsed_candidate = parse_peak_intervals(pd.Series(candidate))
        parse_rate = (
            parsed_candidate[
                ["chrom", "peak_start", "peak_end"]
            ]
            .notna()
            .all(axis=1)
            .mean()
        )

        if parse_rate >= 0.8:
            peak_ids = candidate
        else:
            peak_ids = adata_atac.var_names.astype(str).to_numpy()
    else:
        peak_ids = adata_atac.var_names.astype(str).to_numpy()

    parsed = parse_peak_intervals(pd.Series(peak_ids))

    invalid = parsed[
        ["chrom", "peak_start", "peak_end"]
    ].isna().any(axis=1)

    if invalid.any():
        print(
            f"Warning: invalid ATAC peak coordinates: "
            f"{invalid.sum():,}/{len(parsed):,}"
        )
        print(pd.Series(peak_ids).loc[invalid].head(10).tolist())

    result = pd.DataFrame(
        {
            "peak_id_input": peak_ids,
            "peak_id_canonical": parsed["peak_id_canonical"].to_numpy(),
            "chrom": parsed["chrom"].to_numpy(),
            "peak_start": parsed["peak_start"].to_numpy(),
            "peak_end": parsed["peak_end"].to_numpy(),
            "nonzero_cell_count": nonzero_cell_count,
            "fraction_accessible": fraction_accessible,
            "mean_raw_count": mean_raw_count,
            "pseudobulk_count": pseudobulk_count,
            "pseudobulk_cpm": pseudobulk_cpm,
            "mean_normalized": mean_normalized,
        }
    )

    for column in [
        "highly_variable",
        "means",
        "dispersions",
        "dispersions_norm",
        "feature_types",
        "embed_statue",
    ]:
        if column in adata_atac.var.columns:
            result[column] = adata_atac.var[column].to_numpy()

    result = result.dropna(
        subset=["chrom", "peak_start", "peak_end"]
    ).copy()

    result["peak_start"] = result["peak_start"].astype(int)
    result["peak_end"] = result["peak_end"].astype(int)

    result = result[
        result["peak_end"] > result["peak_start"]
    ].copy()

    valid_metrics = {
        "mean_normalized",
        "fraction_accessible",
        "pseudobulk_cpm",
        "mean_raw_count",
    }

    if ACCESSIBILITY_METRIC not in valid_metrics:
        raise ValueError(
            f"ACCESSIBILITY_METRIC must be one of: {sorted(valid_metrics)}"
        )

    result["peak_accessibility"] = pd.to_numeric(
        result[ACCESSIBILITY_METRIC],
        errors="coerce",
    )

    if LOG1P_ACCESSIBILITY:
        result["peak_accessibility"] = np.log1p(
            result["peak_accessibility"]
        )

    result["peak_length"] = (
        result["peak_end"]
        - result["peak_start"]
    )

    return (
        result.drop_duplicates(
            subset=["peak_id_canonical"],
            keep="first",
        )
        .reset_index(drop=True)
    )


def merge_peak_attention_with_accessibility(
    peak_accessibility: pd.DataFrame,
    ranked_peaks: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    merged = peak_accessibility.merge(
        ranked_peaks[
            [
                "peak_id_canonical",
                "peak_attention",
                "peak_attention_rank",
            ]
        ],
        on="peak_id_canonical",
        how="left",
        validate="one_to_one",
    )

    merged["is_ranked_peak"] = merged["peak_attention"].notna()
    unmatched_ranked = ranked_peaks[
        ~ranked_peaks["peak_id_canonical"].isin(
            set(peak_accessibility["peak_id_canonical"])
        )
    ].copy()

    print(
        f"Ranked peaks matched to ATAC: "
        f"{merged['is_ranked_peak'].sum():,}/"
        f"{len(ranked_peaks):,}"
    )

    return merged, unmatched_ranked


# ============================================================
# 14. Human target-cell RNA expression
# ============================================================

def add_lookup_value(
    lookup: Dict[str, list[int]],
    key: str,
    index: int,
) -> None:
    key = clean_string(key)

    if not key:
        return

    lookup.setdefault(key, []).append(index)


def build_rna_feature_lookup(
    adata_rna: ad.AnnData,
) -> tuple[
    Dict[str, list[int]],
    Dict[str, list[int]],
]:
    gene_id_lookup: Dict[str, list[int]] = {}
    gene_symbol_lookup: Dict[str, list[int]] = {}

    for index, value in enumerate(
        adata_rna.var_names.astype(str)
    ):
        add_lookup_value(
            gene_id_lookup,
            strip_ensembl_version(value),
            index,
        )
        add_lookup_value(
            gene_symbol_lookup,
            value.upper(),
            index,
        )

    for column in [
        "gene_id",
        "gene_ids",
        "ensembl_id",
        "ensembl_ids",
    ]:
        if column not in adata_rna.var.columns:
            continue

        for index, value in enumerate(
            adata_rna.var[column].astype(str).to_numpy()
        ):
            add_lookup_value(
                gene_id_lookup,
                strip_ensembl_version(value),
                index,
            )

    for column in [
        "gene_symbol",
        "gene_symbols",
        "gene_name",
        "gene_names",
        "feature_name",
        "features",
    ]:
        if column not in adata_rna.var.columns:
            continue

        for index, value in enumerate(
            adata_rna.var[column].astype(str).to_numpy()
        ):
            add_lookup_value(
                gene_symbol_lookup,
                value.strip().upper(),
                index,
            )

    gene_id_lookup = {
        key: sorted(set(indices))
        for key, indices in gene_id_lookup.items()
    }
    gene_symbol_lookup = {
        key: sorted(set(indices))
        for key, indices in gene_symbol_lookup.items()
    }

    return gene_id_lookup, gene_symbol_lookup


def calculate_human_ex_gene_expression(
    adata_rna: ad.AnnData,
    ranked_human_genes: pd.DataFrame,
) -> pd.DataFrame:
    ex_mask = select_excitatory_mask(
        obs=adata_rna.obs,
        celltype_column=CELLTYPE_COLUMN,
        dataset_name="Human RNA",
    )

    if RNA_COUNTS_LAYER in adata_rna.layers:
        matrix = adata_rna.layers[RNA_COUNTS_LAYER][ex_mask, :]
        print(f"RNA matrix source: layers['{RNA_COUNTS_LAYER}']")
    else:
        matrix = adata_rna.X[ex_mask, :]
        print("Warning: RNA counts layer is absent; using adata.X.")

    if sp.issparse(matrix):
        matrix = matrix.tocsr().astype(np.float64)
        cell_totals = np.asarray(matrix.sum(axis=1)).ravel()
    else:
        matrix = np.asarray(matrix, dtype=np.float64)
        cell_totals = matrix.sum(axis=1)

    scale_factors = np.divide(
        RNA_TARGET_SUM,
        cell_totals,
        out=np.zeros_like(cell_totals, dtype=float),
        where=cell_totals > 0,
    )

    gene_id_lookup, symbol_lookup = build_rna_feature_lookup(
        adata_rna
    )

    records: list[dict] = []

    for gene in ranked_human_genes.itertuples(index=False):
        gene_id = strip_ensembl_version(gene.gene_id)
        gene_symbol = clean_string(gene.gene_symbol)
        symbol_upper = gene_symbol.upper()

        matched_indices: list[int] = []
        match_method = ""

        if gene_id in gene_id_lookup:
            matched_indices = gene_id_lookup[gene_id]
            match_method = "gene_id"
        elif symbol_upper in symbol_lookup:
            matched_indices = symbol_lookup[symbol_upper]
            match_method = "gene_symbol"

        if not matched_indices:
            records.append(
                {
                    "human_gene_id": gene_id,
                    "human_gene_symbol": gene_symbol,
                    "human_gene_attention": gene.human_gene_attention,
                    "human_gene_rank": gene.human_gene_rank,
                    "rna_match_status": "unmatched",
                    "rna_match_method": "",
                    "rna_n_features": 0,
                    RNA_EXPRESSION_COLUMN: np.nan,
                    "human_ex_fraction_expressing": np.nan,
                    "human_ex_pseudobulk_count": np.nan,
                }
            )
            continue

        selected = matrix[:, matched_indices]

        raw_per_cell = np.asarray(
            selected.sum(axis=1)
        ).ravel()

        normalized_per_cell = raw_per_cell * scale_factors
        log_normalized_per_cell = np.log1p(normalized_per_cell)

        records.append(
            {
                "human_gene_id": gene_id,
                "human_gene_symbol": gene_symbol,
                "human_gene_attention": gene.human_gene_attention,
                "human_gene_rank": gene.human_gene_rank,
                "rna_match_status": "matched",
                "rna_match_method": match_method,
                "rna_n_features": len(matched_indices),
                RNA_EXPRESSION_COLUMN: float(
                    np.mean(log_normalized_per_cell)
                ),
                "human_ex_fraction_expressing": float(
                    np.mean(raw_per_cell > 0)
                ),
                "human_ex_pseudobulk_count": float(
                    np.sum(raw_per_cell)
                ),
            }
        )

    result = pd.DataFrame(records)

    result = assign_quantiles(
        frame=result,
        value_column="human_gene_attention",
        output_column="gene_attention_quantile",
    )

    return result


# ============================================================
# 15. Metagene bin construction
# ============================================================

def build_bins_for_gene(
    gene_start: int,
    gene_end: int,
    strand: str,
) -> pd.DataFrame:
    if gene_end <= gene_start:
        raise ValueError(
            f"Invalid gene interval: {gene_start}-{gene_end}"
        )

    if strand not in {"+", "-"}:
        raise ValueError(f"Invalid strand: {strand}")

    tss = gene_start if strand == "+" else gene_end
    tes = gene_end if strand == "+" else gene_start

    records: list[dict] = []

    upstream_edges = np.linspace(
        0,
        FLANK_SIZE,
        UPSTREAM_BINS + 1,
    )

    for index in range(UPSTREAM_BINS):
        relative_left = int(round(upstream_edges[index]))
        relative_right = int(round(upstream_edges[index + 1]))

        if strand == "+":
            genomic_start = tss - relative_right
            genomic_end = tss - relative_left
        else:
            genomic_start = tss + relative_left
            genomic_end = tss + relative_right

        records.append(
            {
                "bin_index": index,
                "region": "upstream",
                "genomic_start": genomic_start,
                "genomic_end": genomic_end,
            }
        )

    body_edges = np.linspace(
        gene_start,
        gene_end,
        GENE_BODY_BINS + 1,
    )

    for body_index in range(GENE_BODY_BINS):
        genomic_index = (
            body_index
            if strand == "+"
            else GENE_BODY_BINS - body_index - 1
        )

        genomic_start = int(
            round(body_edges[genomic_index])
        )
        genomic_end = int(
            round(body_edges[genomic_index + 1])
        )

        if genomic_end <= genomic_start:
            genomic_end = genomic_start + 1

        records.append(
            {
                "bin_index": UPSTREAM_BINS + body_index,
                "region": "gene_body",
                "genomic_start": genomic_start,
                "genomic_end": genomic_end,
            }
        )

    downstream_edges = np.linspace(
        0,
        FLANK_SIZE,
        DOWNSTREAM_BINS + 1,
    )

    for index in range(DOWNSTREAM_BINS):
        relative_left = int(round(downstream_edges[index]))
        relative_right = int(round(downstream_edges[index + 1]))

        if strand == "+":
            genomic_start = tes + relative_left
            genomic_end = tes + relative_right
        else:
            genomic_start = tes - relative_right
            genomic_end = tes - relative_left

        records.append(
            {
                "bin_index": (
                    UPSTREAM_BINS
                    + GENE_BODY_BINS
                    + index
                ),
                "region": "downstream",
                "genomic_start": genomic_start,
                "genomic_end": genomic_end,
            }
        )

    bins = pd.DataFrame(records)

    bins["genomic_start"] = bins[
        "genomic_start"
    ].clip(lower=0)

    bins["bin_length"] = (
        bins["genomic_end"]
        - bins["genomic_start"]
    )

    bins = bins[
        bins["bin_length"] > 0
    ].copy()

    bins["normalized_position"] = (
        bins["bin_index"] + 0.5
    )

    return bins


def calculate_gene_profile(
    gene: pd.Series,
    chromosome_peaks: pd.DataFrame,
    peak_set_name: str,
) -> pd.DataFrame:
    bins = build_bins_for_gene(
        gene_start=int(gene["gene_start"]),
        gene_end=int(gene["gene_end"]),
        strand=str(gene["strand"]),
    )

    region_start = int(bins["genomic_start"].min())
    region_end = int(bins["genomic_end"].max())

    candidate_peaks = chromosome_peaks[
        (
            chromosome_peaks["peak_end"]
            > region_start
        )
        &
        (
            chromosome_peaks["peak_start"]
            < region_end
        )
    ]

    accessibility_signal = np.zeros(
        len(bins),
        dtype=float,
    )
    peak_attention_signal = np.zeros(
        len(bins),
        dtype=float,
    )
    peak_presence = np.zeros(
        len(bins),
        dtype=int,
    )
    peak_counts = np.zeros(
        len(bins),
        dtype=int,
    )
    peak_coverage_bp = np.zeros(
        len(bins),
        dtype=int,
    )

    if not candidate_peaks.empty:
        peak_starts = candidate_peaks[
            "peak_start"
        ].to_numpy(dtype=int)
        peak_ends = candidate_peaks[
            "peak_end"
        ].to_numpy(dtype=int)
        accessibility_values = candidate_peaks[
            "peak_accessibility"
        ].to_numpy(dtype=float)

        if "peak_attention" in candidate_peaks.columns:
            attention_values = (
                candidate_peaks["peak_attention"]
                .fillna(0)
                .to_numpy(dtype=float)
            )
        else:
            attention_values = np.zeros(
                len(candidate_peaks),
                dtype=float,
            )

        for local_index, bin_row in enumerate(
            bins.itertuples(index=False)
        ):
            overlap_start = np.maximum(
                peak_starts,
                int(bin_row.genomic_start),
            )
            overlap_end = np.minimum(
                peak_ends,
                int(bin_row.genomic_end),
            )
            overlap_bp = np.maximum(
                0,
                overlap_end - overlap_start,
            )
            overlapping = overlap_bp > 0

            if not overlapping.any():
                continue

            accessibility_signal[local_index] = (
                np.sum(
                    accessibility_values[overlapping]
                    * overlap_bp[overlapping]
                )
                / float(bin_row.bin_length)
            )

            peak_attention_signal[local_index] = (
                np.sum(
                    attention_values[overlapping]
                    * overlap_bp[overlapping]
                )
                / float(bin_row.bin_length)
            )

            peak_presence[local_index] = 1
            peak_counts[local_index] = int(overlapping.sum())
            peak_coverage_bp[local_index] = int(
                np.minimum(
                    int(bin_row.bin_length),
                    overlap_bp[overlapping].sum(),
                )
            )

    bins["peak_accessibility_signal"] = accessibility_signal
    bins["peak_attention_signal"] = peak_attention_signal
    bins["peak_presence"] = peak_presence
    bins["overlapping_peak_count"] = peak_counts
    bins["peak_coverage_bp"] = peak_coverage_bp
    bins["peak_coverage_fraction"] = (
        bins["peak_coverage_bp"]
        / bins["bin_length"]
    )

    bins["peak_set"] = peak_set_name
    bins["macaque_gene_id"] = gene["macaque_gene_id"]
    bins["macaque_gene_symbol"] = gene["macaque_gene_symbol"]
    bins["chrom"] = gene["chrom"]
    bins["strand"] = gene["strand"]
    bins["gene_length"] = gene["gene_length"]
    bins["human_gene_attention"] = gene["human_gene_attention"]
    bins["gene_attention_quantile"] = gene[
        "gene_attention_quantile"
    ]

    return bins


def calculate_all_profiles(
    genes: pd.DataFrame,
    peaks: pd.DataFrame,
    peak_set_name: str,
) -> pd.DataFrame:
    peaks_by_chromosome = {
        chromosome: (
            current.sort_values("peak_start")
            .reset_index(drop=True)
        )
        for chromosome, current in peaks.groupby(
            "chrom",
            sort=False,
        )
    }

    empty_peak_table = pd.DataFrame(
        columns=peaks.columns
    )

    profiles: list[pd.DataFrame] = []
    total_genes = len(genes)

    for processed_index, (_, gene) in enumerate(
        genes.iterrows(),
        start=1,
    ):
        chromosome_peaks = peaks_by_chromosome.get(
            gene["chrom"],
            empty_peak_table,
        )

        profiles.append(
            calculate_gene_profile(
                gene=gene,
                chromosome_peaks=chromosome_peaks,
                peak_set_name=peak_set_name,
            )
        )

        if (
            processed_index == 1
            or processed_index % 25 == 0
            or processed_index == total_genes
        ):
            print(
                f"[{peak_set_name}] processed genes: "
                f"{processed_index:,}/{total_genes:,}"
            )

    if not profiles:
        raise ValueError("No metagene profiles were generated.")

    return pd.concat(profiles, ignore_index=True)


def summarize_profiles(
    profile_df: pd.DataFrame,
) -> pd.DataFrame:
    summary = (
        profile_df
        .groupby(
            [
                "peak_set",
                "gene_attention_quantile",
                "bin_index",
                "normalized_position",
                "region",
            ],
            observed=True,
        )
        .agg(
            mean_peak_accessibility=(
                "peak_accessibility_signal",
                "mean",
            ),
            median_peak_accessibility=(
                "peak_accessibility_signal",
                "median",
            ),
            standard_deviation=(
                "peak_accessibility_signal",
                "std",
            ),
            mean_peak_attention=(
                "peak_attention_signal",
                "mean",
            ),
            mean_peak_presence=(
                "peak_presence",
                "mean",
            ),
            mean_peak_count=(
                "overlapping_peak_count",
                "mean",
            ),
            mean_peak_coverage_fraction=(
                "peak_coverage_fraction",
                "mean",
            ),
            n_genes=(
                "macaque_gene_id",
                "nunique",
            ),
        )
        .reset_index()
    )

    summary["standard_deviation"] = (
        summary["standard_deviation"]
        .fillna(0)
    )

    summary["standard_error"] = (
        summary["standard_deviation"]
        / np.sqrt(summary["n_genes"])
    )

    return summary


# ============================================================
# 16. Profile plotting
# ============================================================

def set_metagene_axis(ax: plt.Axes) -> None:
    tss_position = UPSTREAM_BINS
    tes_position = UPSTREAM_BINS + GENE_BODY_BINS
    total_bins = (
        UPSTREAM_BINS
        + GENE_BODY_BINS
        + DOWNSTREAM_BINS
    )

    ax.axvline(
        tss_position,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )
    ax.axvline(
        tes_position,
        linewidth=0.8,
        linestyle="--",
        alpha=0.7,
    )

    flank_label = (
        f"{FLANK_SIZE / 1000:g} kb"
        if FLANK_SIZE >= 1000
        else f"{FLANK_SIZE} bp"
    )

    ax.set_xlim(0, total_bins)
    ax.set_xticks(
        [0, tss_position, tes_position, total_bins]
    )
    ax.set_xticklabels(
        [
            f"−{flank_label}",
            "TSS",
            "TES",
            f"+{flank_label}",
        ]
    )
    ax.set_xlabel(
        "Relative position across macaque ortholog gene region"
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def draw_profile_on_axis(
    ax: plt.Axes,
    summary: pd.DataFrame,
    title: str,
) -> None:
    quantile_order = [
        f"Q{i}"
        for i in range(1, N_GENE_QUANTILES + 1)
    ]

    for quantile in quantile_order:
        current = (
            summary[
                summary["gene_attention_quantile"]
                == quantile
            ]
            .sort_values("bin_index")
        )

        if current.empty:
            continue

        x = current[
            "normalized_position"
        ].to_numpy(dtype=float)
        y = current[
            "mean_peak_accessibility"
        ].to_numpy(dtype=float)
        sem = current[
            "standard_error"
        ].to_numpy(dtype=float)

        if SMOOTH_PROFILE:
            y_plot = rolling_smooth(y, SMOOTH_WINDOW)
            sem_plot = rolling_smooth(sem, SMOOTH_WINDOW)
        else:
            y_plot = y
            sem_plot = sem

        n_genes = int(current["n_genes"].max())

        ax.plot(
            x,
            y_plot,
            linewidth=2,
            label=f"{quantile} (n={n_genes})",
        )

        if SHOW_SEM:
            ax.fill_between(
                x,
                np.maximum(0, y_plot - sem_plot),
                y_plot + sem_plot,
                alpha=0.12,
                linewidth=0,
            )

    set_metagene_axis(ax)

    ax.set_ylabel(
        f"Mean macaque {CURRENT_CELL_TYPE_DISPLAY} peak accessibility"
    )
    ax.set_title(title)
    ax.legend(
        title=(
            f"Human {CURRENT_CELL_TYPE_DISPLAY} gene attention\n"
            "Q1 low → Q4 high"
        ),
        frameon=False,
    )


def save_single_profile(
    summary: pd.DataFrame,
    title: str,
    output_prefix: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(8.5, 5.5))

    draw_profile_on_axis(
        ax=ax,
        summary=summary,
        title=title,
    )

    plt.tight_layout()

    pdf_file = output_prefix.with_suffix(".pdf")
    png_file = output_prefix.with_suffix(".png")

    plt.savefig(pdf_file, bbox_inches="tight")
    plt.savefig(
        png_file,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    print(f"Saved: {pdf_file}")
    print(f"Saved: {png_file}")


# ============================================================
# 17. Human RNA expression statistics and plotting
# ============================================================

def summarize_human_rna_expression(
    genes_rna: pd.DataFrame,
) -> pd.DataFrame:
    data = genes_rna.dropna(
        subset=[
            "gene_attention_quantile",
            RNA_EXPRESSION_COLUMN,
        ]
    ).copy()

    quantile_order = [
        f"Q{i}"
        for i in range(1, N_GENE_QUANTILES + 1)
    ]

    data["gene_attention_quantile"] = pd.Categorical(
        data["gene_attention_quantile"],
        categories=quantile_order,
        ordered=True,
    )

    summary = (
        data
        .groupby(
            "gene_attention_quantile",
            observed=True,
        )
        .agg(
            n_genes=(
                "human_gene_id",
                "nunique",
            ),
            mean_expression=(
                RNA_EXPRESSION_COLUMN,
                "mean",
            ),
            median_expression=(
                RNA_EXPRESSION_COLUMN,
                "median",
            ),
            standard_deviation=(
                RNA_EXPRESSION_COLUMN,
                "std",
            ),
            expression_q25=(
                RNA_EXPRESSION_COLUMN,
                lambda values: values.quantile(0.25),
            ),
            expression_q75=(
                RNA_EXPRESSION_COLUMN,
                lambda values: values.quantile(0.75),
            ),
            mean_gene_attention=(
                "human_gene_attention",
                "mean",
            ),
        )
        .reset_index()
    )

    summary["standard_deviation"] = (
        summary["standard_deviation"]
        .fillna(0)
    )
    summary["standard_error"] = (
        summary["standard_deviation"]
        / np.sqrt(summary["n_genes"])
    )

    return summary


def compute_human_rna_pairwise_tests(
    genes_rna: pd.DataFrame,
) -> pd.DataFrame:
    quantile_order = [
        f"Q{i}"
        for i in range(1, N_GENE_QUANTILES + 1)
    ]

    data = genes_rna.dropna(
        subset=[
            "gene_attention_quantile",
            RNA_EXPRESSION_COLUMN,
        ]
    ).copy()

    groups = {
        quantile: data.loc[
            data["gene_attention_quantile"] == quantile,
            RNA_EXPRESSION_COLUMN,
        ].to_numpy(dtype=float)
        for quantile in quantile_order
    }

    reference = f"Q{N_GENE_QUANTILES}"
    comparisons = [
        (quantile, reference)
        for quantile in quantile_order[:-1]
    ]

    records: list[dict] = []
    raw_p_values: list[float] = []

    for group1, group2 in comparisons:
        values1 = groups[group1]
        values2 = groups[group2]

        if len(values1) == 0 or len(values2) == 0:
            statistic = np.nan
            p_value = np.nan
        else:
            statistic, p_value = stats.mannwhitneyu(
                values1,
                values2,
                alternative="two-sided",
            )

        records.append(
            {
                "group1": group1,
                "group2": group2,
                "n_group1": len(values1),
                "n_group2": len(values2),
                "mean_group1": (
                    float(np.mean(values1))
                    if len(values1) > 0
                    else np.nan
                ),
                "mean_group2": (
                    float(np.mean(values2))
                    if len(values2) > 0
                    else np.nan
                ),
                "median_group1": (
                    float(np.median(values1))
                    if len(values1) > 0
                    else np.nan
                ),
                "median_group2": (
                    float(np.median(values2))
                    if len(values2) > 0
                    else np.nan
                ),
                "mannwhitney_u": statistic,
                "p_value": p_value,
            }
        )

        raw_p_values.append(
            p_value if not pd.isna(p_value) else 1.0
        )

    result = pd.DataFrame(records)

    if not result.empty:
        result["p_adj_bh"] = benjamini_hochberg(
            raw_p_values
        )
        result["significance"] = result[
            "p_adj_bh"
        ].map(pvalue_to_stars)

    return result


def add_significance_annotation(
    ax: plt.Axes,
    x1: float,
    x2: float,
    y: float,
    h: float,
    text: str,
) -> None:
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + h, y + h, y],
        linewidth=1.0,
    )
    ax.text(
        (x1 + x2) / 2,
        y + h,
        text,
        ha="center",
        va="bottom",
        fontsize=10,
    )


def plot_human_rna_expression(
    genes_rna: pd.DataFrame,
    summary: pd.DataFrame,
    output_prefix: Path,
) -> pd.DataFrame:
    quantile_order = [
        f"Q{i}"
        for i in range(1, N_GENE_QUANTILES + 1)
    ]

    data = genes_rna.dropna(
        subset=[
            "gene_attention_quantile",
            RNA_EXPRESSION_COLUMN,
        ]
    ).copy()

    data["gene_attention_quantile"] = pd.Categorical(
        data["gene_attention_quantile"],
        categories=quantile_order,
        ordered=True,
    )

    expression_arrays = [
        data.loc[
            data["gene_attention_quantile"] == quantile,
            RNA_EXPRESSION_COLUMN,
        ].to_numpy(dtype=float)
        for quantile in quantile_order
    ]

    pairwise_stats = compute_human_rna_pairwise_tests(
        data
    )

    pairwise_stats.to_csv(
        HUMAN_RNA_STATS_OUTPUT,
        index=False,
    )

    fig, ax = plt.subplots(figsize=(7.2, 5.8))
    positions = np.arange(1, len(quantile_order) + 1)

    ax.boxplot(
        expression_arrays,
        positions=positions,
        widths=0.55,
        patch_artist=False,
        showfliers=False,
        medianprops={"linewidth": 1.5},
        boxprops={"linewidth": 1.2},
        whiskerprops={"linewidth": 1.0},
        capprops={"linewidth": 1.0},
    )

    rng = np.random.default_rng(42)

    for position, values in zip(
        positions,
        expression_arrays,
    ):
        if len(values) == 0:
            continue

        jitter = rng.uniform(
            -0.18,
            0.18,
            size=len(values),
        )

        ax.scatter(
            np.full(
                len(values),
                position,
                dtype=float,
            )
            + jitter,
            values,
            s=18,
            alpha=0.55,
            linewidths=0,
        )

    summary_indexed = (
        summary
        .set_index("gene_attention_quantile")
        .reindex(quantile_order)
    )

    labels: list[str] = []
    for quantile in quantile_order:
        if quantile in summary_indexed.index:
            n_genes = summary_indexed.loc[
                quantile,
                "n_genes",
            ]

            if pd.notna(n_genes):
                labels.append(
                    f"{quantile}\n(n={int(n_genes)})"
                )
                continue

        labels.append(quantile)

    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_xlabel(
        f"Human {CURRENT_CELL_TYPE_DISPLAY} gene-attention quartile"
    )
    ax.set_ylabel(
        (
            "Mean log-normalized RNA expression in human "
            f"{CURRENT_CELL_TYPE_DISPLAY} cells"
        )
    )
    ax.set_title(
        (
            f"Human {CURRENT_CELL_TYPE_DISPLAY} RNA expression "
            "across gene-attention quartiles"
        )
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    y_values = data[RNA_EXPRESSION_COLUMN].to_numpy(dtype=float)
    y_min = float(np.nanmin(y_values))
    y_max = float(np.nanmax(y_values))
    y_range = max(y_max - y_min, 1e-6)

    bar_height = 0.03 * y_range
    bar_step = 0.08 * y_range
    current_y = y_max + 0.06 * y_range

    position_map = dict(
        zip(quantile_order, positions)
    )

    for row in pairwise_stats.itertuples(index=False):
        if row.significance == "ns":
            continue

        add_significance_annotation(
            ax=ax,
            x1=position_map[row.group1],
            x2=position_map[row.group2],
            y=current_y,
            h=bar_height,
            text=row.significance,
        )
        current_y += bar_step

    valid_arrays = [
        array
        for array in expression_arrays
        if len(array) > 0
    ]

    if len(valid_arrays) >= 2:
        _, kw_p = stats.kruskal(*valid_arrays)

        ax.text(
            0.02,
            0.98,
            f"Kruskal-Wallis p = {kw_p:.2e}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
        )

    ax.set_ylim(
        y_min - 0.05 * y_range,
        current_y + 0.10 * y_range,
    )

    plt.tight_layout()

    pdf_file = output_prefix.with_suffix(".pdf")
    png_file = output_prefix.with_suffix(".png")

    plt.savefig(pdf_file, bbox_inches="tight")
    plt.savefig(
        png_file,
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    print(f"Saved: {pdf_file}")
    print(f"Saved: {png_file}")
    print(f"Saved: {HUMAN_RNA_STATS_OUTPUT}")

    return pairwise_stats


# ============================================================
# 18. Validation helpers
# ============================================================

def report_chromosome_overlap(
    genes: pd.DataFrame,
    peaks: pd.DataFrame,
) -> None:
    gene_chroms = set(genes["chrom"].dropna().astype(str))
    peak_chroms = set(peaks["chrom"].dropna().astype(str))
    overlap = sorted(gene_chroms & peak_chroms)

    print("\nChromosome validation:")
    print(f"  Gene chromosomes: {sorted(gene_chroms)}")
    print(f"  Peak chromosomes: {sorted(peak_chroms)}")
    print(f"  Overlap: {overlap}")

    if not overlap:
        raise ValueError(
            "No chromosome overlap between macaque GFF genes "
            "and ATAC peaks."
        )


def filter_standard_chromosomes(
    frame: pd.DataFrame,
    chromosome_column: str = "chrom",
) -> pd.DataFrame:
    result = frame.copy()

    if KEEP_STANDARD_CHROMOSOMES:
        result = result[
            result[chromosome_column].isin(
                MACAQUE_STANDARD_CHROMS
            )
        ].copy()

    if EXCLUDE_MITOCHONDRIAL:
        result = result[
            result[chromosome_column] != "chrM"
        ].copy()

    return result


# ============================================================
# 19. Main
# ============================================================

def configure_analysis(config: dict) -> None:
    global CURRENT_ANALYSIS_NAME
    global CURRENT_CELL_TYPE
    global CURRENT_CELLTYPE_ALIASES
    global CURRENT_RNA_SAMPLE
    global CURRENT_ATAC_SAMPLE
    global CURRENT_CELL_TYPE_DISPLAY
    global CURRENT_CELL_TYPE_FILE_LABEL
    global MACAQUE_GENE_RANK_FILE
    global HUMAN_GENE_RANK_FILE
    global MACAQUE_PEAK_RANK_FILE
    global MACAQUE_ATAC_H5AD_FILE
    global HUMAN_RNA_H5AD_FILE
    global OUTPUT_DIR

    CURRENT_ANALYSIS_NAME = str(config["analysis_name"])
    CURRENT_CELL_TYPE = str(config["cell_type"]).upper()
    CURRENT_RNA_SAMPLE = str(config["rna_sample"])
    CURRENT_ATAC_SAMPLE = str(config["atac_sample"])
    CURRENT_CELL_TYPE_FILE_LABEL = str(config["cell_type_file_label"])

    if CURRENT_CELL_TYPE not in CELL_TYPE_ALIASES:
        raise ValueError(
            f"Unsupported cell type: {CURRENT_CELL_TYPE}. "
            f"Available: {sorted(CELL_TYPE_ALIASES)}"
        )

    CURRENT_CELL_TYPE_DISPLAY = CELL_TYPE_DISPLAY_NAMES[
        CURRENT_CELL_TYPE
    ]

    CURRENT_CELLTYPE_ALIASES = {
        value.strip().lower()
        for value in CELL_TYPE_ALIASES[CURRENT_CELL_TYPE]
    }

    MACAQUE_GENE_RANK_FILE = Path(config["macaque_gene_rank_file"])
    HUMAN_GENE_RANK_FILE = Path(config["human_gene_rank_file"])
    MACAQUE_PEAK_RANK_FILE = Path(config["macaque_peak_rank_file"])
    MACAQUE_ATAC_H5AD_FILE = Path(config["macaque_atac_h5ad_file"])
    HUMAN_RNA_H5AD_FILE = Path(config["human_rna_h5ad_file"])

    # Keep every result set separate:
    # cell type / RNA sample / ATAC sample
    OUTPUT_DIR = (
        BATCH_OUTPUT_DIR
        / CURRENT_CELL_TYPE_FILE_LABEL
        / f"RNA_{CURRENT_RNA_SAMPLE}"
        / f"ATAC_{CURRENT_ATAC_SAMPLE}"
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    configure_output_paths()


def run_single_analysis() -> None:
    print("=" * 88)
    print(f"Paired RNA–ATAC analysis: {CURRENT_ANALYSIS_NAME} | {CURRENT_CELL_TYPE}")
    print("=" * 88)

    input_files = [
        MACAQUE_GENE_RANK_FILE,
        HUMAN_GENE_RANK_FILE,
        MACAQUE_PEAK_RANK_FILE,
        MACAQUE_GFF_FILE,
        MACAQUE_ATAC_H5AD_FILE,
        HUMAN_RNA_H5AD_FILE,
    ]

    print("\nInput files:")
    for path in input_files:
        require_file(path)
        print(f"  {path}")

    # --------------------------------------------------------
    # 1. Mapped macaque gene ranking
    # --------------------------------------------------------

    print("\n[1] Loading mapped macaque gene ranking")

    macaque_ranked_genes = load_ranked_genes(
        path=MACAQUE_GENE_RANK_FILE,
        species_prefix="macaque",
        score_output_name="human_gene_attention",
    )

    print(
        f"Mapped macaque ranked genes: "
        f"{len(macaque_ranked_genes):,}"
    )
    print(
        macaque_ranked_genes.head(10).to_string(
            index=False
        )
    )

    # --------------------------------------------------------
    # 2. Mmul_10 GFF
    # --------------------------------------------------------

    print("\n[2] Loading Mmul_10 GFF")

    gff_coordinates, chromosome_map = (
        load_macaque_gene_coordinates_from_gff(
            MACAQUE_GFF_FILE
        )
    )

    chromosome_map.to_csv(
        SEQID_CHROMOSOME_MAP_OUTPUT,
        index=False,
    )

    print(
        f"GFF gene records: "
        f"{len(gff_coordinates):,}"
    )
    print(
        f"Sequence-ID chromosome mappings: "
        f"{len(chromosome_map):,}"
    )
    print(f"Saved: {SEQID_CHROMOSOME_MAP_OUTPUT}")

    # --------------------------------------------------------
    # 3. Match macaque genes to GFF
    # --------------------------------------------------------

    print("\n[3] Matching mapped macaque genes to Mmul_10 GFF")

    genes, unmatched_genes = match_macaque_genes_to_gff(
        ranked_genes=macaque_ranked_genes,
        coordinates=gff_coordinates,
    )

    genes = filter_standard_chromosomes(genes)

    if genes.empty:
        raise ValueError(
            "No genes remain after chromosome filtering."
        )

    genes.to_csv(
        MAPPED_GENE_COORDINATE_OUTPUT,
        index=False,
    )

    unmatched_genes.to_csv(
        UNMATCHED_GENE_OUTPUT,
        index=False,
    )

    print(
        f"Matched macaque genes retained: "
        f"{len(genes):,}"
    )
    print(
        f"Unmatched macaque genes: "
        f"{len(unmatched_genes):,}"
    )
    print(f"Saved: {MAPPED_GENE_COORDINATE_OUTPUT}")
    print(f"Saved: {UNMATCHED_GENE_OUTPUT}")

    print("\nGene-attention quartiles:")
    print(
        genes["gene_attention_quantile"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    # --------------------------------------------------------
    # 4. Ranked macaque peaks
    # --------------------------------------------------------

    print("\n[4] Loading macaque peak-attention ranking")

    ranked_peaks = load_ranked_peaks(
        MACAQUE_PEAK_RANK_FILE
    )

    ranked_peaks = filter_standard_chromosomes(
        ranked_peaks
    )

    print(
        f"Valid ranked peaks: "
        f"{len(ranked_peaks):,}"
    )
    # --------------------------------------------------------
    # 5. Macaque target-cell ATAC accessibility
    # --------------------------------------------------------

    print("\n[5] Loading macaque ATAC h5ad")

    adata_atac = ad.read_h5ad(
        MACAQUE_ATAC_H5AD_FILE
    )

    print(adata_atac)
    print("ATAC var columns:")
    print(adata_atac.var.columns.tolist())
    print("ATAC var_names examples:")
    print(adata_atac.var_names[:10].tolist())

    print(f"\n[6] Calculating macaque {CURRENT_CELL_TYPE_DISPLAY} peak accessibility")

    all_peaks = calculate_macaque_ex_peak_accessibility(
        adata_atac
    )

    all_peaks = filter_standard_chromosomes(
        all_peaks
    )

    report_chromosome_overlap(
        genes=genes,
        peaks=all_peaks,
    )

    (
        all_peaks,
        unmatched_ranked_peaks,
    ) = merge_peak_attention_with_accessibility(
        peak_accessibility=all_peaks,
        ranked_peaks=ranked_peaks,
    )

    all_peaks = (
        all_peaks.sort_values(
            "peak_accessibility",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    all_peaks.to_csv(
        ALL_PEAK_ACCESSIBILITY_OUTPUT,
        index=False,
    )
    unmatched_ranked_peaks.to_csv(
        UNMATCHED_PEAK_RANK_OUTPUT,
        index=False,
    )

    print(
        f"All valid ATAC peaks: "
        f"{len(all_peaks):,}"
    )
    print(f"Saved: {ALL_PEAK_ACCESSIBILITY_OUTPUT}")
    print(f"Saved: {UNMATCHED_PEAK_RANK_OUTPUT}")

    del adata_atac
    gc.collect()

    # --------------------------------------------------------
    # 7. Metagene profiles
    # --------------------------------------------------------

    print("\n[7] Calculating all-peak metagene profiles")

    all_profile = calculate_all_profiles(
        genes=genes,
        peaks=all_peaks,
        peak_set_name="All peaks",
    )

    all_profile.to_csv(
        ALL_PROFILE_MATRIX_OUTPUT,
        index=False,
    )

    all_summary = summarize_profiles(
        all_profile
    )

    all_summary.to_csv(
        ALL_PROFILE_SUMMARY_OUTPUT,
        index=False,
    )

    save_single_profile(
        summary=all_summary,
        title=(
            f"Macaque {CURRENT_CELL_TYPE_DISPLAY} accessibility: all peaks, "
            f"grouped by human {CURRENT_CELL_TYPE_DISPLAY} gene attention"
        ),
        output_prefix=ALL_PROFILE_FIGURE_PREFIX,
    )

    # --------------------------------------------------------
    # 9. Human RNA expression validation
    # --------------------------------------------------------

    print("\n[10] Loading human gene ranking")

    human_ranked_genes = load_ranked_genes(
        path=HUMAN_GENE_RANK_FILE,
        species_prefix="human",
        score_output_name="human_gene_attention",
    )

    print(
        f"Human ranked genes: "
        f"{len(human_ranked_genes):,}"
    )

    print("\n[11] Loading human RNA h5ad")

    adata_rna = ad.read_h5ad(
        HUMAN_RNA_H5AD_FILE
    )

    print(adata_rna)
    print("RNA var columns:")
    print(adata_rna.var.columns.tolist())
    print("RNA var_names examples:")
    print(adata_rna.var_names[:10].tolist())

    print(f"\n[12] Calculating human {CURRENT_CELL_TYPE_DISPLAY} RNA expression")

    human_rna_genes = calculate_human_ex_gene_expression(
        adata_rna=adata_rna,
        ranked_human_genes=human_ranked_genes,
    )

    human_rna_summary = summarize_human_rna_expression(
        human_rna_genes
    )

    human_rna_genes.to_csv(
        HUMAN_RNA_GENE_OUTPUT,
        index=False,
    )
    human_rna_summary.to_csv(
        HUMAN_RNA_SUMMARY_OUTPUT,
        index=False,
    )

    plot_human_rna_expression(
        genes_rna=human_rna_genes,
        summary=human_rna_summary,
        output_prefix=HUMAN_RNA_FIGURE_PREFIX,
    )

    print(
        "\nHuman RNA matching status:"
    )
    print(
        human_rna_genes["rna_match_status"]
        .value_counts(dropna=False)
        .to_string()
    )

    print(f"Saved: {HUMAN_RNA_GENE_OUTPUT}")
    print(f"Saved: {HUMAN_RNA_SUMMARY_OUTPUT}")

    del adata_rna
    gc.collect()

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    print("\n" + "=" * 88)
    print("Done")
    print("=" * 88)

    print(
        f"Matched macaque genes: "
        f"{len(genes):,}"
    )
    print(
        f"All macaque ATAC peaks: "
        f"{len(all_peaks):,}"
    )
    print(
        f"Human ranked genes with RNA records: "
        f"{len(human_rna_genes):,}"
    )
    print(
        f"ATAC accessibility metric: "
        f"{ACCESSIBILITY_METRIC}"
    )
    print(
        f"Output directory:\n{OUTPUT_DIR}"
    )


def main() -> None:
    BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    summary_records: list[dict] = []

    for index, config in enumerate(ANALYSIS_CONFIGS, start=1):
        configure_analysis(config)

        print("\n" + "#" * 100)
        print(
            f"[{index}/{len(ANALYSIS_CONFIGS)}] "
            f"{CURRENT_ANALYSIS_NAME} | cell type={CURRENT_CELL_TYPE}"
        )
        print("#" * 100)

        try:
            run_single_analysis()
            status = "SUCCESS"
            error_message = ""
        except Exception as exc:
            status = "FAILED"
            error_message = f"{type(exc).__name__}: {exc}"
            print("\n" + "!" * 88, file=sys.stderr)
            print(
                f"FAILED: {CURRENT_ANALYSIS_NAME} | {error_message}",
                file=sys.stderr,
            )
            print("!" * 88, file=sys.stderr)

        summary_records.append(
            {
                "analysis_name": CURRENT_ANALYSIS_NAME,
                "cell_type": CURRENT_CELL_TYPE,
                "rna_sample": CURRENT_RNA_SAMPLE,
                "atac_sample": CURRENT_ATAC_SAMPLE,
                "human_gene_rank_file": str(HUMAN_GENE_RANK_FILE),
                "macaque_gene_rank_file": str(MACAQUE_GENE_RANK_FILE),
                "macaque_peak_rank_file": str(MACAQUE_PEAK_RANK_FILE),
                "human_rna_h5ad_file": str(HUMAN_RNA_H5AD_FILE),
                "macaque_atac_h5ad_file": str(MACAQUE_ATAC_H5AD_FILE),
                "output_dir": str(OUTPUT_DIR),
                "status": status,
                "error": error_message,
            }
        )

    summary = pd.DataFrame(summary_records)

    print("\n" + "=" * 100)
    print(summary[["analysis_name", "cell_type", "rna_sample", "atac_sample", "status", "error"]].to_string(index=False))
    print("=" * 100)

    if (summary["status"] == "FAILED").any():
        failed = summary.loc[summary["status"] == "FAILED", "analysis_name"].tolist()
        raise RuntimeError(f"Failed analyses: {failed}")


if __name__ == "__main__":
    main()
